## Generate 1D trajectory data by SDE sampling 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

### Define the potential $V$ and its gradient


In [ ]:
# potential V, one-dimensional
def V(x):
    y1 = x**8
    y2 = 0.8 * np.exp(-80 * x**2)
    y3 = 0.55 * np.exp(-80 * (x-0.5)**2)
    y4 = 0.3 * np.exp(-80 * (x+0.5)**2)

    y = 2 * (y1 + y2 + y3 + y4)

    return y

# gradient of V
def gradV(x):
    y1 = 8 * x**7 
    y2 = - 0.8 * 160 * x * np.exp(-80 * x**2)
    y3 = - 0.55 * 160 * (x - 0.5) * np.exp(-80 * (x-0.5)**2)
    y4 = - 0.3 * 160 * (x + 0.5) * np.exp(-80 * (x+0.5)**2)

    y = 2 * (y1 + y2 + y3 + y4)

    return y

#### function to sample the process

In [ ]:
# sample the SDE using Euler-Maruyama scheme
def sample(beta=1.0, dt=0.001, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
    X = 0.0
    traj = []
    tlist = []
    for i in range(N):
        traj.append(X)
        tlist.append(dt*i)        
        b = rng.normal()
        X = X - gradV(X) * dt + np.sqrt(2 * dt/beta) * b

    return np.array(tlist), np.array(traj)  

#### parameters

In [ ]:
# coefficient in SDE
beta = 2.0
# step-size 
dt = 0.005
# number of sampling steps 
N = 50000
# range of the domain 
xmin, xmax = -1.0, 1.0

### plot the potential $V(x)$ and the invariant density (boltzmann density)
$$\pi(x) = \frac{1}{Z}\mathrm{e}^{-\beta V(x)},$$  
where $$Z = \int_{\mathbb{R}} \mathrm{e}^{-\beta V(x)} dx$$ is a normalizing constant.

In [ ]:
# uniform grid on [xmin, xmax]
xvec = np.linspace(xmin, xmax, 101)

# potential on grid
pot_vals = V(xvec)

# compute invariant density
density_unnormalized = np.exp(-beta * pot_vals)
# normalizing constant Z
z = np.sum(density_unnormalized) * (xmax-xmin) / 100
# normalize to get densitz
density_pi = density_unnormalized / z

fig = plt.figure(figsize=(12,4))
ax = fig.add_subplot(1, 2, 1)
# plot V
ax.plot(xvec, pot_vals)
ax.set_xlabel(r'x')
ax.set_title(r'V')

ax = fig.add_subplot(1, 2, 2)
# plot invariant density
ax.plot(xvec, density_pi)
ax.set_xlabel(r'x')
ax.set_title(r'invarint density')

### get trajectory data by sampling

In [ ]:
tvec, traj = sample(beta, dt=dt, N=N)

### display the sampled trajectory and verify that the trajectory is long enough.

When the simulation time is long enough, the empirical density of the trajectory data should match the invariant density (ergodic theorm). 


In [ ]:
fig = plt.figure(figsize=(12,4))
ax = fig.add_subplot(1, 2, 1)

# plot trajectory vs time
ax.plot(tvec, traj, alpha=0.5)
ax.set_ylim([xmin, xmax])
ax.set_xlabel(r'time')
ax.set_ylabel(r'x')
ax.set_title('trajectory')

ax1 = fig.add_subplot(1, 2, 2)

# plot empirical density of the trajectory data
ax1.hist(traj, 50, density=True, label='empirical density')

# plot the invariant density
ax1.plot(xvec, density_pi, label='invarint density')

ax1.set_title('impirical and invariant density')
ax1.legend()